### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable
from sklearn.metrics import r2_score 

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
effective_fr_file = os.path.join(sim_results_folder, 'first_session_effective_firing_rates.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [4]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()


effective_results = np.load(effective_fr_file)
effective_vector = effective_results.mean(axis=0).flatten()
reference = effective_results[:,:,-1]
delta_results = effective_results - np.expand_dims(reference, 2)
delta_vector = delta_results.mean(axis=0).flatten()


# Get the data for the first session
data = load_data(emp_at_file)
data = get_session_data(data, 1)


# Map synchrony values to each condition in DataFrame
data['Synchrony'] = data['Condition'].apply(lambda x: sync_results_vector[x-1])
data['EffectiveFiring'] = data['Condition'].apply(lambda x: effective_vector[x-1])
data['DeltaFiring'] = data['Condition'].apply(lambda x: delta_vector[x-1])

data_valid = data[data['ContrastHeterogeneity'] != 1].copy()

# Z-score relevant columns
data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony', 'EffectiveFiring'])
data_valid = zscore_data(data_valid, ['Synchrony', 'EffectiveFiring', 'DeltaFiring'])

### Define statistical models

In [5]:
# Features-only hierarchical logistic regression
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Model synchrony (mechanism)
model_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + (1 + Synchrony | SubjectID)",
    data=data,
    family="bernoulli"
)

model_effective = bmb.Model(
    "Correct ~ 1 + EffectiveFiring + (1 + EffectiveFiring | SubjectID)",
    data=data,
    family="bernoulli"
)

model_delta = bmb.Model(
    "Correct ~ 1 + DeltaFiring + (1 + DeltaFiring | SubjectID)",
    data=data_valid,
    family="bernoulli"
)

Are the factors that determine synchrony among coupled oscillators (frequency detuning and coupling strength) predictive of human ability to segregate a rectangular figure from its background in texture stimuli? 

In [6]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneit

In [7]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['less', 'less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.999 |
|            GridCoarseness            |    less   | 0.999 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.998 |
+--------------------------------------+-----------+-------+

Odds ratios:
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.557 |    0.408     |     0.741     |
|            GridCoarseness            | 0.766 |    0.665     |     0.876     |
| ContrastHeterogeneity:GridCoarseness | 1.267 |    1.118     |     1.435     |
+--------------------------------------+----

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.595,0.148,-0.891,-0.298,0.003,0.002,2736.0,3562.0,1.0
GridCoarseness,-0.269,0.069,-0.404,-0.131,0.001,0.001,4125.0,4336.0,1.0
ContrastHeterogeneity:GridCoarseness,0.235,0.061,0.116,0.363,0.001,0.001,4187.0,3868.0,1.0


Does the synchronization behavior of a biophysical model of V1 predict human ability to segregate a rectangular figure from its background in texture stimuli?

In [8]:
idata_sync = model_sync.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 185 seconds.


In [9]:
predictors = ["Synchrony"]
directions = ['greater']

posterior = posterior_table(idata_sync, predictors, directions)
odds_ratios = OR_table(idata_sync, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_sync, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+-----------+-----------+-------+
| Predictor | direction |   P   |
+-----------+-----------+-------+
| Synchrony |  greater  | 0.998 |
+-----------+-----------+-------+

Odds ratios:
+-----------+-------+--------------+---------------+
| Predictor |  Mean | Lower (2.5%) | Upper (97.5%) |
+-----------+-------+--------------+---------------+
| Synchrony | 2.187 |    1.384     |     3.328     |
+-----------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Synchrony,0.758,0.219,0.333,1.206,0.005,0.005,2243.0,2973.0,1.0


Does firing rate explain figure-ground perception? Does it do equally well as synchrony?

Step 1: Does pure firing rate in the figure explain figure-ground perception?

In [10]:
idata_effective = model_effective.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, EffectiveFiring, 1|SubjectID_sigma, 1|SubjectID_offset, EffectiveFiring|SubjectID_sigma, EffectiveFiring|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 114 seconds.


In [11]:
predictors = ["EffectiveFiring"]
directions = ['greater']

posterior = posterior_table(idata_effective, predictors, directions)
odds_ratios = OR_table(idata_effective, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_effective, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+-----------------+-----------+-------+
|    Predictor    | direction |   P   |
+-----------------+-----------+-------+
| EffectiveFiring |  greater  | 0.941 |
+-----------------+-----------+-------+

Odds ratios:
+-----------------+-------+--------------+---------------+
|    Predictor    |  Mean | Lower (2.5%) | Upper (97.5%) |
+-----------------+-------+--------------+---------------+
| EffectiveFiring | 1.073 |    0.975     |     1.177     |
+-----------------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
EffectiveFiring,0.069,0.046,-0.025,0.163,0.001,0.001,3850.0,3128.0,1.0


Does the difference in firing between figure and background explain figure-ground perception? 
This assumes some additional downstream mechanisms that compares firing rates in different stimulus regions (not explicitly modeled)

In [12]:
idata_delta = model_delta.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, DeltaFiring, 1|SubjectID_sigma, 1|SubjectID_offset, DeltaFiring|SubjectID_sigma, DeltaFiring|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 127 seconds.


In [13]:
predictors = ["DeltaFiring"]
directions = ['less']

posterior = posterior_table(idata_delta, predictors, directions)
odds_ratios = OR_table(idata_delta, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_delta, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+-------------+-----------+-------+
|  Predictor  | direction |   P   |
+-------------+-----------+-------+
| DeltaFiring |    less   | 1.000 |
+-------------+-----------+-------+

Odds ratios:
+-------------+-------+--------------+---------------+
|  Predictor  |  Mean | Lower (2.5%) | Upper (97.5%) |
+-------------+-------+--------------+---------------+
| DeltaFiring | 0.583 |    0.458     |     0.725     |
+-------------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
DeltaFiring,-0.547,0.115,-0.78,-0.321,0.002,0.003,2363.0,3261.0,1.0


Compare firing rate difference to synchrony. First refit synchrony model only on valid part of data.

In [14]:
model_sync_valid = bmb.Model(
    "Correct ~ 1 + Synchrony + (1 + Synchrony | SubjectID)",
    data=data_valid,
    family="bernoulli"
)

idata_sync_valid = model_sync_valid.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 157 seconds.


In [ ]:
model_effective_valid = bmb.Model(
    "Correct ~ 1 + EffectiveFiring + (1 + EffectiveFiring | SubjectID)",
    data=data_valid,
    family="bernoulli"
)

idata_effective_valid = model_effective_valid.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, EffectiveFiring, 1|SubjectID_sigma, 1|SubjectID_offset, EffectiveFiring|SubjectID_sigma, EffectiveFiring|SubjectID_offset]


In [ ]:
az.summary(idata_effective_valid, var_names=predictors, hdi_prob=0.95)

In [15]:
# Compare models (LOO)
az.compare({
    "synchrony": idata_sync_valid,
    "delta firing rate": idata_delta,
}, method="BB-pseudo-BMA")

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
synchrony,0,-2771.272224,15.991700,0.000000,0.90605,30.173919,0.000000,False,log
delta firing rate,1,-2790.296239,14.888943,19.024014,0.09395,30.853518,14.132063,False,log


### Design Analysis

In [16]:
rng = np.random.default_rng(1709026616) # Seed for reproducibility
num_simulations = 50
num_subjects_list = [4, 6, 8, 10]

num_available_draws = len(az.extract(idata_features, var_names=["Intercept"]).to_dataframe())

results = []
for num_subjects in num_subjects_list:
    for sim in range(num_simulations):
        draw_index = rng.integers(num_available_draws)
        df_simulated, true_betas = simulate_dataset_from_draw(
            idata_features, data, draw_index, num_subjects, rng=rng
        )
        sim_result = analyze_simulated(df_simulated, true_betas)
        sim_result["num_subjects"] = num_subjects
        sim_result["simulation"] = sim
        results.append(sim_result)

results_df = pd.DataFrame(results)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 97 seconds.
There were 8 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:

In [22]:
summary = summarize_design_analysis(results_df)

print(summary)

+--------------+-------------+-------------+----------------------+------------------------------+-----------------+------------------+--------------------+--------------------+--------------------------+----------------------------+----------------------------+-------------------------------------+
| Num Subjects | Detected CH | Detected GC | Detected Interaction | contrast_heterogeneity_typeS | Type S error GC | Type S error INT |  Type M error CH   |  Type M error GC   | Type M error Interaction | Probability (one-sided) CH | Probability (one-sided) GC | Probability (one-sided) Interaction |
+--------------+-------------+-------------+----------------------+------------------------------+-----------------+------------------+--------------------+--------------------+--------------------------+----------------------------+----------------------------+-------------------------------------+
|     4.0      |     0.58    |     0.68    |         0.44         |             0.0              